In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
dim_drivers_df = (
    spark.table(f"{catalog_name}.{silver_schema}.drivers")
    .filter(F.col("batch_id") == v_batch_id)
    .select("driver_id", "driver_name", "date_of_birth", "nationality")
)

In [0]:
dim_drivers_df = (
    dim_drivers_df
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
    )

In [0]:
if not spark.catalog.tableExists(target_table):
    (
        dim_drivers_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("t")
        .merge(
            dim_drivers_df.alias("d"),
            "t.driver_id = d.driver_id"
        )
        .whenMatchedUpdate(
            set={
                "driver_name": "d.driver_name",
                "date_of_birth": "d.date_of_birth",
                "nationality": "d.nationality",
                "updated_at": "d.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )